# CIGALE Decomposition Validation: Bayesian Alpha Calibration

The earlier validation notebooks compared a single point estimate -
`median(alpha_fit / alpha_theory)` with a bootstrap CI - to test whether the
paper's formula `alpha_theory = fracAGN / (1 - fracAGN)` (from
`F_composite = F_gal + alpha * S * F_AGN`, i.e. AGN's share of total
integrated flux = `alpha/(1+alpha)`) correctly predicts the `alpha` needed
to reconstruct each real galaxy's full SED.

**This notebook upgrades that check into a proper Bayesian regression**:
instead of one ratio number, it fits the full population relationship
```
log10(alpha_fit) = beta0 + beta1 * log10(alpha_theory) + N(0, sigma^2)
```
via MCMC (`emcee`), giving posterior distributions over the intercept
(`beta0`), slope (`beta1`), and intrinsic scatter (`sigma`) - so instead of
"the median ratio is 0.99", the claim becomes "here is the full credible
region for how alpha_fit relates to alpha_theory, and here is whether the
null hypothesis (`beta0=0, beta1=1`, i.e. alpha_theory is unbiased) is
consistent with the data." The fit is repeated for all three AGN templates
from prior notebooks (Type1, Type2, Matched) so the comparison also answers:
**does the best-performing template (Matched) show tighter posterior
scatter than the two fixed templates**, as an independent Bayesian check on
the earlier frequentist finding that Matched reconstructs real galaxies
best?

New dependencies added to this project for this notebook: `emcee` (MCMC
ensemble sampler) and `corner` (posterior corner plots) - this project
previously did all statistics by hand in `scipy`/`numpy`
(`docs/figure9_10_bootstrap_methodology.md`), with no probabilistic
programming library.

This notebook is additive: it does not modify any of
`CIGALE_Decomposition_Validation.ipynb`,
`CIGALE_Decomposition_Validation_GeometryFix.ipynb`, or
`CIGALE_Decomposition_Validation_WeightedBlend.ipynb`. It reuses their
cached results (`alpha_fit_Type1`/`alpha_fit_Type2` from the first
notebook's cache, the per-galaxy geometry from the second notebook's cache)
and computes one new quantity not previously cached: `alpha_fit_Matched`.

In [ ]:
import sys
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import emcee
import corner

sys.path.append(os.path.abspath('..'))

from src import config
from glass import data_io, composite_math, photometry, visualization, analysis

plt.style.use('default')
visualization.apply_pasa_style()
os.makedirs(config.PROCESSED_DATA_DIR, exist_ok=True)

BAYES_OUTPUT_DIR = os.path.join(config.PROCESSED_DATA_DIR, 'cigale_bayesian_alpha')
os.makedirs(BAYES_OUTPUT_DIR, exist_ok=True)

VALIDATION_DIR = os.path.join(config.PROCESSED_DATA_DIR, 'cigale_theoretical_validation')
GEOMETRY_FIX_DIR = os.path.join(config.PROCESSED_DATA_DIR, 'cigale_geometry_fix')

TEMPLATE_COLORS = {'Type1': '#1A6FB5', 'Type2': '#CC2929', 'Matched': '#2E8B57'}

## 1. Data: alpha_theory vs alpha_fit for three templates

`alpha_theory = fracAGN / (1 - fracAGN)` is the same for every template (it
only depends on CIGALE's `fracAGN`, not on which AGN template is used to
test it). `alpha_fit_Type1`/`alpha_fit_Type2` are reused directly from
`CIGALE_Decomposition_Validation.ipynb`'s cache. `alpha_fit_Matched` is new:
the same closed-form least-squares calculation
(`alpha_fit = sum[(full-host)*scaled_agn] / sum[scaled_agn**2]`, on the
wavelength grid aligned/scaled exactly as `create_composite_sed` would),
but using each galaxy's own CIGALE-matched SKIRTOR geometry (i=30 or i=70,
selected via `agn.i` - see
`CIGALE_Decomposition_Validation_GeometryFix.ipynb` for the full diagnosis
of why only these two geometries appear) instead of a template applied
uniformly to every galaxy.

In [ ]:
cigale_csv = os.path.join(config.RAW_DATA_DIR, 'full_zfourge_decomposed', 'zfourge_full_final.csv')
agn_frac_csv = os.path.join(config.RAW_DATA_DIR, 'full_zfourge_decomposed', 'agn_fractions.csv')
geometry_csv = os.path.join(GEOMETRY_FIX_DIR, 'agn_geometry.csv')
validation_csv = os.path.join(VALIDATION_DIR, 'reconstruction_summary.csv')

for req, msg in [(geometry_csv, 'CIGALE_Decomposition_Validation_GeometryFix.ipynb'),
                  (validation_csv, 'CIGALE_Decomposition_Validation.ipynb')]:
    if not os.path.exists(req):
        raise FileNotFoundError(f"{req} not found - run {msg} first to build its cache.")

df_cig = pd.read_csv(cigale_csv, low_memory=False)
z_col = 'zpk_x' if 'zpk_x' in df_cig.columns else 'zpk'
df_cig = df_cig.merge(pd.read_csv(agn_frac_csv), on='ID', how='left')
df_agn = df_cig[df_cig['fracAGN'] > 0.0].reset_index(drop=True)
df_agn = df_agn.merge(pd.read_csv(geometry_csv), on='ID', how='left')

alpha_ref = pd.read_csv(validation_csv)[['ID', 'alpha_theory', 'alpha_fit_Type1', 'alpha_fit_Type2']]
df_agn = df_agn.merge(alpha_ref, on='ID', how='left')

print(f"Population: {len(df_agn)} galaxies with alpha_theory and alpha_fit_Type1/Type2 already available.")


def _fits_path(gid, field):
    fits_num = gid.split('_', 1)[1]
    return os.path.join(config.RAW_DATA_DIR, 'full_zfourge_decomposed',
                         f"{field.lower()}_best_models_fits", f"{fits_num}_best_model.fits")

In [ ]:
skirtor_dir = os.path.join(config.RAW_DATA_DIR, 'Templates', 'Skirtor')
MATCHED_GEOMETRY_PARAMS = {'optical_depth': 7, 'p': 1, 'q': 1, 'opening_angle': 40, 'radius_ratio': 20}
MATCHED_TEMPLATES = {
    30: data_io.read_skirtor_model(skirtor_dir, inclination=30, **MATCHED_GEOMETRY_PARAMS),
    70: data_io.read_skirtor_model(skirtor_dir, inclination=70, **MATCHED_GEOMETRY_PARAMS),
}

FLUX_FLOOR_FRACTION = 1e-4


def alpha_fit_matched(gid, field, z, fracAGN, agn_i):
    '''Closed-form least-squares alpha_fit using each galaxy's own CIGALE-matched SKIRTOR template.'''
    matched_key = int(round(agn_i)) if np.isfinite(agn_i) else None
    if matched_key not in MATCHED_TEMPLATES:
        return np.nan
    path = _fits_path(gid, field)
    if not os.path.exists(path):
        return np.nan

    full_sed = data_io.read_cigale_best_model(path, redshift=z, restframe=True)
    wl_full = full_sed['lambda (Angstroms)'].values.astype(float)
    full_L = full_sed['L_lambda_total'].values.astype(float)
    peak_L = np.nanmax(full_L)
    floor = FLUX_FLOOR_FRACTION * peak_L if peak_L > 0 else 0.0

    host_sed = analysis.decompose_cigale_sed(full_sed, target='host')
    agn_template = MATCHED_TEMPLATES[matched_key]

    wl_al, agn_al, host_al = composite_math.adjust_wavelength_range(
        agn_template['lambda (Angstroms)'].values, agn_template['Total Flux (erg/s/cm^2/Angstrom)'].values,
        host_sed['lambda (Angstroms)'].values, host_sed['Total Flux (erg/s/cm^2/Angstrom)'].values)
    S = composite_math.compute_scaling_factor(wl_al, agn_al, wl_al, host_al)
    scaled_agn = agn_al * S
    full_al = np.interp(wl_al, wl_full, full_L, left=np.nan, right=np.nan)
    m = np.isfinite(full_al) & np.isfinite(host_al) & np.isfinite(scaled_agn) & (full_al > floor)
    denom = np.sum(scaled_agn[m] ** 2)
    return np.sum((full_al[m] - host_al[m]) * scaled_agn[m]) / denom if denom > 0 else np.nan


MATCHED_ALPHA_CSV = os.path.join(BAYES_OUTPUT_DIR, 'alpha_fit_matched.csv')
if os.path.exists(MATCHED_ALPHA_CSV):
    matched_alpha_df = pd.read_csv(MATCHED_ALPHA_CSV)
    print(f"Loaded cached alpha_fit_Matched for {len(matched_alpha_df)} galaxies.")
else:
    rows = []
    t0 = time.time()
    for _, r in df_agn.iterrows():
        af = alpha_fit_matched(r['ID'], r['field'], r[z_col], r['fracAGN'], r['i'])
        rows.append({'ID': r['ID'], 'alpha_fit_Matched': af})
        if len(rows) % 1000 == 0:
            print(f"  {len(rows)}/{len(df_agn)} processed ({time.time() - t0:.0f}s elapsed)")
    matched_alpha_df = pd.DataFrame(rows)
    matched_alpha_df.to_csv(MATCHED_ALPHA_CSV, index=False)
    print(f"Done: {len(matched_alpha_df)} galaxies, {time.time() - t0:.0f}s total.")

df_agn = df_agn.merge(matched_alpha_df, on='ID', how='left')

## 2. The Bayesian model

```
y_i = log10(alpha_fit_i),  x_i = log10(alpha_theory_i)
y_i ~ Normal(beta0 + beta1 * x_i, sigma^2)
```
with weakly-informative flat priors `beta0, beta1 ~ Uniform(-3, 3)`,
`log10(sigma) ~ Uniform(-5, 3)` - deliberately uninformative so the ~6,500
data points dominate the posterior. The null hypothesis "`alpha_theory`
correctly predicts `alpha_fit`, unbiased" is exactly `beta0=0, beta1=1`.

Only galaxies with `alpha_fit > 0` can be log-transformed; a negative or
zero closed-form `alpha_fit` means the least-squares fit found no
consistent positive scaling of that template against the real residual
(effectively "this template doesn't explain this galaxy's AGN excess at
all") - excluded from the fit, with the excluded fraction reported per
template as its own diagnostic (a template that's a poor match for the real
data should have *more* galaxies excluded here).

In [ ]:
def log_prior(theta):
    beta0, beta1, log_sigma = theta
    if -3 < beta0 < 3 and -3 < beta1 < 3 and -5 < log_sigma < 3:
        return 0.0
    return -np.inf


def log_likelihood(theta, x, y):
    beta0, beta1, log_sigma = theta
    sigma = 10 ** log_sigma
    model = beta0 + beta1 * x
    return np.sum(-0.5 * np.log(2 * np.pi * sigma ** 2) - 0.5 * (y - model) ** 2 / sigma ** 2)


def log_posterior(theta, x, y):
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood(theta, x, y)


def fit_mcmc(x, y, nwalkers=32, nsteps=4000, burn=1000, seed=42):
    '''Runs emcee on the (beta0, beta1, log_sigma) log-log regression model, returns the flattened,
    burn-in-discarded chain.'''
    rng = np.random.default_rng(seed)
    # Seed walkers near a quick least-squares estimate for faster convergence.
    beta1_0, beta0_0 = np.polyfit(x, y, 1)
    resid = y - (beta0_0 + beta1_0 * x)
    log_sigma_0 = np.log10(max(np.std(resid), 1e-3))
    p0_center = np.array([beta0_0, beta1_0, log_sigma_0])
    p0 = p0_center + 1e-2 * rng.standard_normal((nwalkers, 3))

    sampler = emcee.EnsembleSampler(nwalkers, 3, log_posterior, args=(x, y))
    sampler.run_mcmc(p0, nsteps, progress=False)
    flat = sampler.get_chain(discard=burn, flat=True)
    return sampler, flat

In [ ]:
TEMPLATES_TO_FIT = ['Type1', 'Type2', 'Matched']
data = {}
results = {}

for tname in TEMPLATES_TO_FIT:
    af = df_agn[f'alpha_fit_{tname}']
    at = df_agn['alpha_theory']
    valid = np.isfinite(af) & np.isfinite(at) & (af > 0) & (at > 0)
    x = np.log10(at[valid].values)
    y = np.log10(af[valid].values)
    data[tname] = (x, y)
    n_total = np.isfinite(af).sum()
    n_excluded = n_total - valid.sum()
    print(f"{tname}: n_valid={valid.sum()}, n_excluded (alpha_fit<=0)={n_excluded} "
          f"({n_excluded / n_total:.1%} of {n_total})")

    t0 = time.time()
    sampler, flat_samples = fit_mcmc(x, y, seed=100 + TEMPLATES_TO_FIT.index(tname))
    accept_frac = np.mean(sampler.acceptance_fraction)
    try:
        tau = sampler.get_autocorr_time(quiet=True)
    except Exception:
        tau = np.array([np.nan, np.nan, np.nan])
    results[tname] = {'flat_samples': flat_samples, 'sampler': sampler,
                       'accept_frac': accept_frac, 'autocorr_time': tau, 'fit_time': time.time() - t0}
    print(f"  MCMC done in {results[tname]['fit_time']:.1f}s; mean acceptance fraction = {accept_frac:.2f}; "
          f"autocorrelation time (beta0,beta1,log_sigma) = {tau}")

## 3. Convergence diagnostics (Figure 1: trace plot)

Shows the `Matched` template's chains as a representative example - all
`nwalkers` chains should mix well and show no long-term drift after
burn-in, and the reported acceptance fractions (~0.2-0.5 is the
conventionally healthy range for `emcee`'s stretch-move sampler) and
autocorrelation times (should be much shorter than the chain length) above
confirm this for all three templates.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(visualization.PASA_WIDE[0], visualization.PASA_WIDE[1] * 1.4), sharex=True)
labels = ['beta0', 'beta1', 'log10(sigma)']
chain = results['Matched']['sampler'].get_chain()
for i in range(3):
    ax = axes[i]
    ax.plot(chain[:, :, i], color='k', alpha=0.15, lw=0.5)
    ax.set_ylabel(labels[i])
axes[-1].set_xlabel('MCMC step')
axes[0].set_title('Matched template: chain trace (all walkers)')

plt.tight_layout()
fig.savefig(os.path.join(BAYES_OUTPUT_DIR, 'Figure1_trace_matched.png'), dpi=300, bbox_inches='tight')
plt.show()

## 4. Posterior corner plots (Figure 2)

One corner plot per template. Dashed lines mark the null hypothesis
(`beta0=0, beta1=1`) - if the posterior contours comfortably contain that
point, `alpha_theory` is statistically an unbiased predictor of
`alpha_fit` for that template; if the contours exclude it, there's a
population-level bias the point-estimate ratio in the earlier notebooks
didn't quantify.

In [ ]:
summary_rows = []
for tname in TEMPLATES_TO_FIT:
    flat_samples = results[tname]['flat_samples']
    fig = corner.corner(flat_samples, labels=['beta0', 'beta1', 'log10(sigma)'],
                         truths=[0, 1, None], truth_color='red',
                         quantiles=[0.16, 0.5, 0.84], show_titles=True, title_fmt='.3f',
                         color=TEMPLATE_COLORS[tname])
    fig.suptitle(f'{tname}: posterior over (beta0, beta1, log10 sigma)', y=1.02)
    fig.savefig(os.path.join(BAYES_OUTPUT_DIR, f'Figure2_corner_{tname}.png'), dpi=200, bbox_inches='tight')
    plt.show()

    b0_lo, b0_med, b0_hi = np.percentile(flat_samples[:, 0], [16, 50, 84])
    b1_lo, b1_med, b1_hi = np.percentile(flat_samples[:, 1], [16, 50, 84])
    ls_lo, ls_med, ls_hi = np.percentile(flat_samples[:, 2], [16, 50, 84])
    summary_rows.append({'template': tname, 'beta0_med': b0_med, 'beta0_lo': b0_lo, 'beta0_hi': b0_hi,
                          'beta1_med': b1_med, 'beta1_lo': b1_lo, 'beta1_hi': b1_hi,
                          'sigma_dex_med': 10 ** ls_med, 'sigma_dex_lo': 10 ** ls_lo, 'sigma_dex_hi': 10 ** ls_hi})
    print(f"{tname}: beta0={b0_med:+.3f} [{b0_lo:+.3f},{b0_hi:+.3f}], "
          f"beta1={b1_med:.3f} [{b1_lo:.3f},{b1_hi:.3f}], "
          f"sigma={10**ls_med:.3f} dex [{10**ls_lo:.3f},{10**ls_hi:.3f}]")

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(os.path.join(BAYES_OUTPUT_DIR, 'posterior_summary.csv'), index=False)

## 5. Posterior predictive check (Figure 3)

Overlays posterior-drawn regression lines on the raw `alpha_fit` vs
`alpha_theory` scatter, for all three templates on one panel - the
Bayesian equivalent of checking whether the fitted line actually tracks
the data, and how the credible band's width compares across templates.

In [ ]:
fig, ax = plt.subplots(figsize=visualization.PASA_WIDE)
x_grid = np.linspace(-2, 2, 100)
rng = np.random.default_rng(0)

for tname in TEMPLATES_TO_FIT:
    x, y = data[tname]
    ax.scatter(10 ** x, 10 ** y, s=2, alpha=0.06, color=TEMPLATE_COLORS[tname])

    flat_samples = results[tname]['flat_samples']
    draws = flat_samples[rng.choice(len(flat_samples), 200, replace=False)]
    y_draws = np.array([b0 + b1 * x_grid for b0, b1, _ in draws])
    lo, med, hi = np.percentile(y_draws, [16, 50, 84], axis=0)
    ax.plot(10 ** x_grid, 10 ** med, color=TEMPLATE_COLORS[tname], lw=2, label=tname)
    ax.fill_between(10 ** x_grid, 10 ** lo, 10 ** hi, color=TEMPLATE_COLORS[tname], alpha=0.2)

ax.plot(10 ** x_grid, 10 ** x_grid, 'k--', lw=1, label='1:1 (null hypothesis)')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('alpha_theory = fracAGN / (1 - fracAGN)')
ax.set_ylabel('alpha_fit (closed-form least squares)')
ax.set_title('Posterior predictive regression, all three templates')
ax.legend(fontsize=8)

plt.tight_layout()
fig.savefig(os.path.join(BAYES_OUTPUT_DIR, 'Figure3_posterior_predictive.png'), dpi=300, bbox_inches='tight')
plt.show()

## 6. Verification and summary

Reports whether the null hypothesis (`beta0=0, beta1=1`) sits inside each
template's 68% credible interval on both parameters simultaneously, and
compares the intrinsic scatter (`sigma`, in dex) across templates - a
direct Bayesian counterpart to the earlier finding that Matched has the
best flux/colour fidelity.

In [ ]:
print("=== Bayesian alpha-calibration summary ===")
for _, row in summary_df.iterrows():
    tname = row['template']
    b0_in = row['beta0_lo'] <= 0 <= row['beta0_hi']
    b1_in = row['beta1_lo'] <= 1 <= row['beta1_hi']
    verdict = 'consistent with alpha_theory (unbiased)' if (b0_in and b1_in) else 'shows a population-level bias'
    print(f"\n{tname}: beta0={row['beta0_med']:+.3f} [{row['beta0_lo']:+.3f},{row['beta0_hi']:+.3f}] "
          f"(contains 0: {b0_in}), beta1={row['beta1_med']:.3f} [{row['beta1_lo']:.3f},{row['beta1_hi']:.3f}] "
          f"(contains 1: {b1_in}) -> {verdict}")
    print(f"  Intrinsic scatter: {row['sigma_dex_med']:.3f} dex [{row['sigma_dex_lo']:.3f},{row['sigma_dex_hi']:.3f}]")

tightest = summary_df.loc[summary_df['sigma_dex_med'].idxmin(), 'template']
print(f"\nTightest posterior scatter: {tightest}")

**Caveat:** this model assumes homoscedastic scatter (one `sigma` for
the whole population) and a single power-law relation across the full
`fracAGN` range. Given the flux-floor and Lyman-continuum handling already
documented in `docs/cigale_recombination_validation_findings.md` §7.3-7.4,
and the very sparse high-`fracAGN` tail (`docs/figure9_10_bootstrap_methodology.md`
notes n=12 at `fracAGN=0.99`), the fitted `sigma` is dominated by the
well-populated low-to-mid `fracAGN` bins; a fully hierarchical model with
per-bin scatter would be a natural next extension if the high-`fracAGN`
tail's behaviour specifically needs quantifying.

This notebook does not modify any prior notebook, script, or `glass`-package
code. `pyproject.toml`/`uv.lock` were updated to add the `emcee` and
`corner` dependencies used here.